# Local Hubble Fit Analysis

This notebook loads the SH0ES Pantheon+ data, verifies the required redshift and distance modulus columns, plots the low-redshift sample, and fits a straight line to estimate the local Hubble flow.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

DATA_PATH = 'Pantheon+SH0ES.dat'

# Load the data using whitespace-delimited parsing and preserve headers
df = pd.read_csv(DATA_PATH, delim_whitespace=True, comment='#', header=0)
df.head()
df.count()

: 

In [ ]:
# Verify that zCMB and a MU-like distance modulus column are present
print('Has zCMB column:', 'zCMB' in df.columns)
print('Has MU column:', 'MU' in df.columns)
print('MU-like columns:', [c for c in df.columns if c.startswith('MU')])

# Use MU_SH0ES as the distance modulus column if MU is not present
if 'MU' not in df.columns and 'MU_SH0ES' in df.columns:
    df = df.rename(columns={'MU_SH0ES': 'MU', 'MU_SH0ES_ERR_DIAG': 'MU_ERR'})

print('\nFinal columns used:')
print('zCMB' in df.columns, 'MU' in df.columns, 'MU_ERR' in df.columns)

In [ ]:
# Select the low-redshift sample up to z = 0.1
mask = df['zCMB'] <= 0.1
df_lowz = df.loc[mask].copy()

z = df_lowz['zCMB'].to_numpy()
mu = df_lowz['MU'].to_numpy()
mu_err = df_lowz['MU_ERR'].to_numpy()

plt.figure(figsize=(9, 6))
plt.errorbar(z, mu, yerr=mu_err, fmt='o', ms=4, alpha=0.7, label='Data (z <= 0.1)')
plt.xlabel('Redshift $z_{athrm{CMB}}$')
plt.ylabel('Distance Modulus $u$')
plt.title('Low-redshift SH0ES Pantheon+ Data')
plt.grid(alpha=0.4, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit a straight-line model to the low-redshift data using curve_fit
def linear_model(z, slope, intercept):
    return slope * z + intercept

initial_guess = [30.0, 30.0]
popt, pcov = curve_fit(linear_model, z, mu, sigma=mu_err, absolute_sigma=True, p0=initial_guess)
slope, intercept = popt
slope_err, intercept_err = np.sqrt(np.diag(pcov))

print(f'Straight-line fit: mu = {slope:.3f} * z + {intercept:.3f}')
print(f'Slope uncertainty: {slope_err:.3f}, intercept uncertainty: {intercept_err:.3f}')

z_fit = np.linspace(0, 0.1, 200)
mu_fit = linear_model(z_fit, slope, intercept)

plt.figure(figsize=(9, 6))
plt.errorbar(z, mu, yerr=mu_err, fmt='o', ms=4, alpha=0.7, label='Data (z <= 0.1)')
plt.plot(z_fit, mu_fit, color='red', lw=2, label='Linear fit')
plt.xlabel('Redshift $z_{athrm{CMB}}$')
plt.ylabel('Distance Modulus $u$')
plt.title('Linear Fit to Low-redshift Distance Modulus')
plt.grid(alpha=0.4, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

## Interpretation

The slope of the fitted line is a crude proxy for the local Hubble flow in the low-redshift regime. 
In a more physical model, the distance modulus is related to the Hubble constant through the luminosity distance formula, but this straight-line fit provides a first-order local estimate from the observed redshift-distance trend.